<a href="https://colab.research.google.com/github/nalgo-intern/xxx/blob/test_b/hello.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
hello="HELLO"
world="WORLD"
print(f'{hello},{world}')

HELLO,WORLD


In [4]:
!pip install fugashi ipadic unidic-lite

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.4/13.4 MB 7.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 MB 11.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 694.9/694.9 kB 11.0 MB/s eta 0:00:00
  Created wheel for ipadic: filename=ipadic-1.0.0-py3-none-any.whl size=13556704 sha256=e134371cfc98c139a888535a3c46dd62b05725bc660e3e12d36312f2975bcd79
  Stored in directory: /root/.cache/pip/wheels/93/8b/55/dd5978a069678c372520847cf84ba2ec539cb41917c00a2206
  Created wheel for unidic-lite: filename=unidic_lite-1.0.8-py3-none-any.whl size=47658817 sha256=423f601060dad2030762fbffe27ab8ebc1f21c658bebb46ccd08f7ead93914c9
  Stored in directory: /root/.cache/pip/wheels/5e/1f/0f/4d43887e5476d956fae828ee9b6687becd5544d68b51ed633d
Successfully built ipadic unidic-lite


In [5]:
import pandas as pd

# データの例（0: 不満/言及なし, 1: 満足）
data = [
    {"text": "料理は絶品ですが、接客がイマイチで待ち時間も長かった。", "taste": 1, "service": 0, "congestion": 0},
    {"text": "店員さんが親切で料理も美味しく、待ち時間ゼロでした！", "taste": 1, "service": 1, "congestion": 1},
    {"text": "値段の割に味が普通。店内が混んでいて落ち着かなかった。", "taste": 0, "service": 0, "congestion": 0},
    # ※本番では数十〜数百件程度のデータを用意
]

df = pd.DataFrame(data)

In [6]:
from transformers import AutoTokenizer
import torch

# 日本語モデルの標準「東北大BERT」を指定
MODEL_NAME = "cl-tohoku/bert-base-japanese-v3"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class ReviewDataset(torch.utils.data.Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.df = df
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        inputs = self.tokenizer(
            row['text'],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        # 3つのラベルをテンソルに変換（多ラベル分類用）
        labels = torch.tensor([row['taste'], row['service'], row['congestion']], dtype=torch.float)

        return {
            'input_ids': inputs['input_ids'].squeeze(0),
            'attention_mask': inputs['attention_mask'].squeeze(0),
            'labels': labels
        }

In [7]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3, # [味, 接客, 混雑] の3つ
    problem_type="multi_label_classification" # 多ラベル分類を指定
)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  447MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: cl-tohoku/bert-base-japanese-v3
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those 

In [8]:
from transformers import Trainer, TrainingArguments

# Datasetの準備
dataset = ReviewDataset(df, tokenizer)

# 学習の設定
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,           # 学習回数（エポック数）
    per_device_train_batch_size=8, # バッチサイズ
    logging_steps=10,
    save_strategy="no",           # 4日間の開発なら保存は最後のみでOK
)

# Trainerの設定と学習開始
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
)

# 学習実行！(Colab GPUなら数十秒〜数分で終わります)
trainer.train()

# 学習済みモデルとトークナイザーを保存
model.save_pretrained("./my_aspect_bert")
tokenizer.save_pretrained("./my_aspect_bert")

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./my_aspect_bert/tokenizer_config.json',
 './my_aspect_bert/vocab.txt',
 './my_aspect_bert/added_tokens.json')

In [10]:
def predict_aspect_score(text):
    # 1. テキストをトークナイズ
    inputs = tokenizer(text, return_tensors="pt", max_length=128, truncation=True, padding=True)

    # 🌟 【修正箇所】入力テンソルをモデルと同じデバイス (CPU or GPU) に送る
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    # 2. 推論実行
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        # Sigmoid関数を通して0.0〜1.0（確率）に変換
        probs = torch.sigmoid(outputs.logits)[0].tolist()

    # 3. パーセント（%）化して返却
    results = {
        "味・品質": f"{round(probs[0] * 100)}%",
        "接客・雰囲気": f"{round(probs[1] * 100)}%",
        "混雑・待ち時間": f"{round(probs[2] * 100)}%"
    }
    return results

# 動作確認（エラーなく動くはずです！）
sample_review = "お肉は柔らかくて最高でした！ただ、接客が雑で少し並びました。"
print(predict_aspect_score(sample_review))

{'味・品質': '75%', '接客・雰囲気': '31%', '混雑・待ち時間': '31%'}


In [11]:
# 1. ライブラリのインストール
!pip install gradio pandas plotly -q

import gradio as gr
import pandas as pd

# 2. 感情分析ロジック（ダミー関数：ここに実際のモデル/LLM処理を入れる）
def analyze_review_ui(review_text):
    if not review_text.strip():
        return "テキストを入力してください。", None, ""

    # 本来はここに機械学習/LLMの計算処理が入る
    # 例としてダミーのパーセントスコアを返す
    scores = {
        "接客・雰囲気": 0.80, # 80%
        "味・品質": 0.63,     # 63%
        "混雑・待ち時間": 0.35 # 35%
    }

    insight_text = "💡 **AIインサイト:**\n味や接客は高く評価されていますが、待ち時間に対する不満（35%）が全体の評価を押し下げている傾向があります。"

    return scores, insight_text

# 3. Gradio UIの構築
with gr.Blocks(title="口コミ要素別 感情分析ツール") as demo:
    gr.Markdown("# 🏨 口コミ要素別 満足度分析ツール")
    gr.Markdown("星評価では見えない「接客」「味」「混雑」などの個別要素を可視化します。")

    with gr.Row():
        with gr.Column():
            # 入力エリア
            input_text = gr.Textbox(
                label="分析する口コミテキスト",
                lines=5,
                placeholder="ここにGoogleマップ等の口コミを貼り付けてください..."
            )
            submit_btn = gr.Button("分析実行", variant="primary")

        with gr.Column():
            # 出力エリア
            gr.Markdown("### 【口コミ要素別 満足度スコア】")
            # gr.Label は辞書型（{"要素名": スコア(0~1)}）を渡すとバー形式で表示してくれる
            output_scores = gr.Label(label="要素別ポジティブ率")
            output_insight = gr.Markdown()

    # ボタン押下時の処理
    submit_btn.click(
        fn=analyze_review_ui,
        inputs=[input_text],
        outputs=[output_scores, output_insight]
    )

# 4. 起動（share=Trueで公開URLを発行）
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://dee296f9eb7d4ab71d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://dee296f9eb7d4ab71d.gradio.live
